# Public repository note

This notebook is an output-cleared code copy. It requires locally authorised data and is not runnable from the public repository alone. Green Street raw data, intermediate files, derived aggregates and outputs are not distributed.


# 04 Source Reconciliation and Indicator Build

This notebook turns the currently available Green Street and OpenLocal/Gavin datasets into analysis-ready indicators.

It does **not** require the pending Green Street 2019-2025 historical POI churn file. The historical POI data will later slot into the workflow as an additional churn/event indicator. For now, the focus is on:

1. building the Green Street vacancy, long-term vacancy and Health Index panel for 2019-2025;
2. aggregating OpenLocal/Gavin quarterly property records into annual retail and office indicators;
3. aligning Green Street and OpenLocal/Gavin at LAD-year level;
4. creating source-validation and coverage checks;
5. exporting analysis-ready restricted intermediate files for later hypothesis testing.

## 1. Setup

Outputs from this notebook are restricted derived files. They should be kept local unless aggregation and disclosure risk are reviewed.

In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)

BASE = Path(os.environ.get("DISSERTATION_WORKSPACE", Path.cwd().resolve()))
OUTPUT_DIR = BASE / "outputs" / "restricted_source_reconciliation"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

BASE, OUTPUT_DIR

## 2. File Configuration

In [ ]:
files = {
    "greenstreet_vacancy_2019_2024": "Greater_London_2019 to 2024 Vacancy - 2026-07-01-1218.csv",
    "greenstreet_vacancy_2025": "Greater_London_2025_Vacancy_2026-07-08-1222.csv",
    "greenstreet_vacancy_2026": "Greater_London_2026_Vacancy_2026-07-01-1222.csv",
    "openlocal_parquet": "2025-12-31-shuting-yang-greenstreet-retail.parquet",
    "lad_boundaries": "Local_Authority_Districts_May_2024_Boundaries_UK_BGC_3503156029110784919.geojson",
    "office_markets": "London_Office_Markets_V1.geojson",
    "prepared_greenstreet_dir": "outputs/restricted_greenstreet_prepared",
}

file_inventory = []
for key, fname in files.items():
    p = BASE / fname
    file_inventory.append({
        "key": key,
        "path": fname,
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / 1024 / 1024, 2) if p.exists() and p.is_file() else np.nan,
    })
file_inventory = pd.DataFrame(file_inventory)
display(file_inventory)

missing = file_inventory.loc[~file_inventory["exists"], "key"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required files/directories: {missing}")

## 3. Helper Functions

These helpers standardise column names, parse mixed date formats, assign points to LADs and office submarkets, and calculate weighted rates.

In [ ]:
def standardise_columns(df):
    out = df.copy()
    out.columns = [str(c).upper() for c in out.columns]
    return out


def parse_mixed_date(series):
    out = pd.to_datetime(series, errors="coerce")
    missing = out.isna() & series.notna()
    if missing.any():
        out.loc[missing] = pd.to_datetime(series.loc[missing], errors="coerce", dayfirst=True)
    return out


def to_numeric(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def points_from_lonlat(df, lon="LONGITUDE", lat="LATITUDE", crs="EPSG:4326"):
    if lon not in df.columns or lat not in df.columns:
        raise KeyError(f"Expected coordinate fields {lon!r} and {lat!r}")
    clean = df.dropna(subset=[lon, lat]).copy()
    return gpd.GeoDataFrame(clean, geometry=gpd.points_from_xy(clean[lon], clean[lat]), crs=crs)


def load_spatial_layers():
    lad = gpd.read_file(BASE / files["lad_boundaries"]).to_crs("EPSG:4326")
    office = gpd.read_file(BASE / files["office_markets"]).to_crs("EPSG:4326")

    submarket_crosswalk = {
        "West End": ["Mayfair", "Soho", "St James's", "Covent Garden", "Fitzrovia", "North of Oxford Street", "Paddington", "Knightsbridge", "Victoria"],
        "City": ["City Core"],
        "Tech Belt & Midtown": ["Midtown", "Bloomsbury", "Clerkenwell", "Euston", "Kings Cross", "Shoreditch", "Camden", "Aldgate & Whitechapel"],
        "Canary Wharf": ["Canary Wharf"],
        "Southbank": ["Southbank", "Waterloo", "Vauxhall, Nine Elms and Battersea"],
    }
    market_to_group = {market: group for group, markets in submarket_crosswalk.items() for market in markets}
    office["study_submarket"] = office["Market"].map(market_to_group).fillna("Outside core / comparison")
    office["inside_core_submarket"] = office["study_submarket"].ne("Outside core / comparison")
    return lad, office


def assign_geographies(points_gdf, lad, office):
    out = gpd.sjoin(
        points_gdf,
        office[["Market", "study_submarket", "inside_core_submarket", "geometry"]],
        how="left",
        predicate="intersects",
    )
    out = out.drop(columns=["index_right"], errors="ignore")
    out["study_submarket"] = out["study_submarket"].fillna("Outside office market polygon")
    out["inside_core_submarket"] = out["inside_core_submarket"].fillna(False).astype(bool)

    out = gpd.sjoin(
        out,
        lad[["LAD24CD", "LAD24NM", "geometry"]],
        how="left",
        predicate="intersects",
    )
    out = out.drop(columns=["index_right"], errors="ignore")
    return out


def weighted_rate(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce").fillna(0)
    denominator = pd.to_numeric(denominator, errors="coerce").fillna(0)
    den_sum = denominator.sum()
    return np.nan if den_sum == 0 else numerator.sum() / den_sum


def safe_pct_change(current, baseline):
    if pd.isna(current) or pd.isna(baseline) or baseline == 0:
        return np.nan
    return (current - baseline) / baseline


lad, office = load_spatial_layers()
print("Loaded LAD polygons:", len(lad))
print("Loaded office market polygons:", len(office))

## 4. Green Street Vacancy, Long-Term Vacancy and Health Panel

The core Green Street longitudinal panel uses 2019-2025. The 2025 file contains additional property types, so the comparable panel keeps `PROPERTY_TYPE == High Street` to align with the 2019-2024 historical file.

In [ ]:
def read_greenstreet_vacancy(path, source_label):
    df = pd.read_csv(path, low_memory=False)
    df = standardise_columns(df)
    df["source_file"] = source_label
    if "DATE_EOM" in df.columns:
        df["DATE_EOM_PARSED"] = parse_mixed_date(df["DATE_EOM"])
        df["year"] = df["DATE_EOM_PARSED"].dt.year
    return df


gs_hist = read_greenstreet_vacancy(BASE / files["greenstreet_vacancy_2019_2024"], "2019-2024 historical vacancy")
gs_2025_all = read_greenstreet_vacancy(BASE / files["greenstreet_vacancy_2025"], "2025 vacancy")

# Keep the comparable High Street subset in 2025 because the 2025 export includes additional property types.
if "PROPERTY_TYPE" in gs_2025_all.columns:
    gs_2025 = gs_2025_all[gs_2025_all["PROPERTY_TYPE"].eq("High Street")].copy()
else:
    gs_2025 = gs_2025_all.copy()

core_numeric_cols = [
    "PROPERTY_ID", "LATITUDE", "LONGITUDE", "NUMBER_UNIT", "NUMBER_UNIT_VACANT", "NUMBER_VACANT_LT",
    "RATE_VACANT", "RATE_VACANT_LT", "SCORE_HEALTH_INDEX",
    "NUMBER_UNIT_CHANGE_OPEN_YOY", "NUMBER_UNIT_CHANGE_CLOSE_YOY", "NUMBER_UNIT_CHANGE_YOY",
]

gs_core = pd.concat([gs_hist, gs_2025], ignore_index=True, sort=False)
gs_core = gs_core[gs_core["year"].between(2019, 2025)].copy()
gs_core = to_numeric(gs_core, core_numeric_cols)

print("2019-2024 historical rows:", len(gs_hist))
print("2025 full export rows:", len(gs_2025_all))
print("2025 comparable High Street rows:", len(gs_2025))
print("Core Green Street panel rows:", len(gs_core))

display(
    gs_core.groupby("year").agg(
        rows=("PROPERTY_ID", "size"),
        properties=("PROPERTY_ID", "nunique"),
        property_types=("PROPERTY_TYPE", lambda s: ", ".join(sorted(map(str, s.dropna().unique()))[:5])),
        total_units=("NUMBER_UNIT", "sum"),
        mean_vacancy=("RATE_VACANT", "mean"),
        mean_health_index=("SCORE_HEALTH_INDEX", "mean"),
    ).reset_index()
)

In [ ]:
gs_points = points_from_lonlat(gs_core)
gs_geo = assign_geographies(gs_points, lad, office)

# Keep a non-spatial property-year table for export.
gs_property_year_panel = gs_geo.drop(columns="geometry", errors="ignore").copy()

print("Green Street property-year rows with coordinates:", len(gs_geo))
print("Rows matched to LAD:", int(gs_geo["LAD24NM"].notna().sum()))
print("Rows inside five core office submarkets:", int(gs_geo["inside_core_submarket"].sum()))

display(
    gs_geo.groupby(["year", "study_submarket"]).agg(
        rows=("PROPERTY_ID", "size"),
        properties=("PROPERTY_ID", "nunique"),
        total_units=("NUMBER_UNIT", "sum"),
    ).reset_index().head(30)
)

In [ ]:
def aggregate_greenstreet_group(g):
    return pd.Series({
        "gs_rows": len(g),
        "gs_properties": g["PROPERTY_ID"].nunique(dropna=True),
        "gs_total_units": pd.to_numeric(g["NUMBER_UNIT"], errors="coerce").sum(),
        "gs_total_vacant_units": pd.to_numeric(g["NUMBER_UNIT_VACANT"], errors="coerce").sum(),
        "gs_weighted_vacancy": weighted_rate(g["NUMBER_UNIT_VACANT"], g["NUMBER_UNIT"]),
        "gs_mean_vacancy": pd.to_numeric(g["RATE_VACANT"], errors="coerce").mean(),
        "gs_total_long_term_vacant_units": pd.to_numeric(g.get("NUMBER_VACANT_LT", np.nan), errors="coerce").sum(),
        "gs_weighted_long_term_vacancy": weighted_rate(g.get("NUMBER_VACANT_LT", pd.Series(np.nan, index=g.index)), g["NUMBER_UNIT"]),
        "gs_mean_long_term_vacancy": pd.to_numeric(g.get("RATE_VACANT_LT", np.nan), errors="coerce").mean(),
        "gs_mean_health_index": pd.to_numeric(g.get("SCORE_HEALTH_INDEX", np.nan), errors="coerce").mean(),
        "gs_opened_units_yoy": pd.to_numeric(g.get("NUMBER_UNIT_CHANGE_OPEN_YOY", np.nan), errors="coerce").sum(),
        "gs_closed_units_yoy": pd.to_numeric(g.get("NUMBER_UNIT_CHANGE_CLOSE_YOY", np.nan), errors="coerce").sum(),
        "gs_net_unit_change_yoy": pd.to_numeric(g.get("NUMBER_UNIT_CHANGE_YOY", np.nan), errors="coerce").sum(),
    })


gs_lad_year = (
    gs_geo.dropna(subset=["LAD24NM"])
    .groupby(["year", "LAD24CD", "LAD24NM"], dropna=False)
    .apply(aggregate_greenstreet_group)
    .reset_index()
    .sort_values(["LAD24NM", "year"])
)

gs_submarket_year = (
    gs_geo
    .groupby(["year", "study_submarket", "inside_core_submarket"], dropna=False)
    .apply(aggregate_greenstreet_group)
    .reset_index()
    .sort_values(["study_submarket", "year"])
)

display(gs_lad_year.head(12))
display(gs_submarket_year.head(12))

## 5. OpenLocal/Gavin Quarterly to Annual Indicators

OpenLocal/Gavin records are quarterly snapshots from 2019 Q1 to 2025 Q4. Annual indicators are calculated as the mean of quarterly snapshot indicators within each year.

In [ ]:
openlocal_cols = [
    "period", "geocode_name", "geocode", "category_group", "uarn", "account_name",
    "occupation_state", "total_floor_area", "rateable_value", "rates_payable", "unadjusted_price",
]

openlocal = pd.read_parquet(BASE / files["openlocal_parquet"], columns=openlocal_cols)
openlocal["period"] = pd.to_datetime(openlocal["period"], errors="coerce")
openlocal["year"] = openlocal["period"].dt.year
openlocal["category_group"] = openlocal["category_group"].astype("string")
openlocal["occupation_state"] = openlocal["occupation_state"].astype("string")

numeric_cols = ["total_floor_area", "rateable_value", "rates_payable", "unadjusted_price"]
openlocal = to_numeric(openlocal, numeric_cols)

openlocal_main = openlocal[
    openlocal["year"].between(2019, 2025)
    & openlocal["category_group"].isin(["RETAIL", "OFFICE"])
].copy()

print("OpenLocal rows loaded:", len(openlocal))
print("OpenLocal retail/office rows:", len(openlocal_main))
print("Periods:", openlocal_main["period"].min(), "to", openlocal_main["period"].max())
display(openlocal_main["category_group"].value_counts(dropna=False).rename("records").to_frame())
display(openlocal_main["occupation_state"].value_counts(dropna=False).rename("records").to_frame())

In [ ]:
def aggregate_openlocal_snapshot(g):
    occupied = g["occupation_state"].eq("OCCUPIED").sum()
    vacant = g["occupation_state"].eq("VACANT").sum()
    valid_occ = occupied + vacant
    floors = g["total_floor_area"].dropna()
    return pd.Series({
        "records": len(g),
        "unique_units": g["uarn"].nunique(dropna=True),
        "accounts": g["account_name"].nunique(dropna=True),
        "occupied_records": occupied,
        "vacant_records": vacant,
        "vacancy_share": np.nan if valid_occ == 0 else vacant / valid_occ,
        "total_floor_area": g["total_floor_area"].sum(),
        "occupied_floor_area": g.loc[g["occupation_state"].eq("OCCUPIED"), "total_floor_area"].sum(),
        "median_floor_area": floors.median() if len(floors) else np.nan,
        "p25_floor_area": floors.quantile(0.25) if len(floors) else np.nan,
        "p75_floor_area": floors.quantile(0.75) if len(floors) else np.nan,
        "small_unit_share_100sqm": (floors <= 100).mean() if len(floors) else np.nan,
        "large_unit_share_1000sqm": (floors >= 1000).mean() if len(floors) else np.nan,
        "total_rateable_value": g["rateable_value"].sum(),
        "median_rateable_value": g["rateable_value"].median(),
        "total_rates_payable": g["rates_payable"].sum(),
        "median_unadjusted_price": g["unadjusted_price"].median(),
    })


snapshot_keys = ["year", "period", "geocode_name", "geocode", "category_group"]
openlocal_snapshot = (
    openlocal_main
    .groupby(snapshot_keys, dropna=False)
    .apply(aggregate_openlocal_snapshot)
    .reset_index()
)

print("Quarterly snapshot indicator rows:", len(openlocal_snapshot))
display(openlocal_snapshot.head(12))

In [ ]:
indicator_cols = [
    "records", "unique_units", "accounts", "occupied_records", "vacant_records", "vacancy_share",
    "total_floor_area", "occupied_floor_area", "median_floor_area", "p25_floor_area", "p75_floor_area",
    "small_unit_share_100sqm", "large_unit_share_1000sqm", "total_rateable_value",
    "median_rateable_value", "total_rates_payable", "median_unadjusted_price",
]

openlocal_annual_long = (
    openlocal_snapshot
    .groupby(["year", "geocode_name", "geocode", "category_group"], dropna=False)
    .agg(
        n_quarters=("period", "nunique"),
        **{col: (col, "mean") for col in indicator_cols}
    )
    .reset_index()
)

def prefix_category(df, category, prefix):
    out = df[df["category_group"].eq(category)].copy()
    rename = {
        col: f"{prefix}_{col}"
        for col in out.columns
        if col not in ["year", "geocode_name", "geocode", "category_group"]
    }
    out = out.rename(columns=rename).drop(columns=["category_group"])
    return out

retail_annual = prefix_category(openlocal_annual_long, "RETAIL", "ol_retail")
office_annual = prefix_category(openlocal_annual_long, "OFFICE", "ol_office")

openlocal_lad_year = retail_annual.merge(
    office_annual,
    on=["year", "geocode_name", "geocode"],
    how="outer",
).sort_values(["geocode_name", "year"])

print("OpenLocal annual LAD-year rows:", len(openlocal_lad_year))
display(openlocal_lad_year.head(12))

In [ ]:
# Add changes relative to the 2019 baseline for selected OpenLocal indicators.
baseline_metrics = [
    "ol_retail_vacancy_share",
    "ol_retail_total_floor_area",
    "ol_retail_unique_units",
    "ol_retail_median_floor_area",
    "ol_retail_total_rateable_value",
    "ol_office_vacancy_share",
    "ol_office_total_floor_area",
    "ol_office_unique_units",
    "ol_office_median_floor_area",
    "ol_office_small_unit_share_100sqm",
    "ol_office_large_unit_share_1000sqm",
]

baseline = openlocal_lad_year[openlocal_lad_year["year"].eq(2019)][
    ["geocode_name", "geocode"] + [m for m in baseline_metrics if m in openlocal_lad_year.columns]
].copy()
baseline = baseline.rename(columns={m: f"{m}_2019" for m in baseline_metrics if m in baseline.columns})

openlocal_lad_year = openlocal_lad_year.merge(baseline, on=["geocode_name", "geocode"], how="left")
for metric in [m for m in baseline_metrics if m in openlocal_lad_year.columns]:
    base = f"{metric}_2019"
    openlocal_lad_year[f"{metric}_change_since_2019"] = openlocal_lad_year[metric] - openlocal_lad_year[base]
    openlocal_lad_year[f"{metric}_pct_change_since_2019"] = [
        safe_pct_change(cur, b)
        for cur, b in zip(openlocal_lad_year[metric], openlocal_lad_year[base])
    ]

display(openlocal_lad_year.head(12))

## 6. Green Street and OpenLocal Reconciliation Panel

This is the main validation/reconciliation object. It does not answer the hypotheses directly; instead, it checks which LAD-year observations have both Green Street retail indicators and OpenLocal/Gavin retail-office indicators.

In [ ]:
validation_panel = gs_lad_year.merge(
    openlocal_lad_year,
    left_on=["year", "LAD24CD"],
    right_on=["year", "geocode"],
    how="inner",
    validate="many_to_one",
)

validation_panel["has_gs_core_coverage"] = (
    validation_panel["gs_properties"].ge(1)
    & validation_panel["gs_total_units"].gt(0)
)
validation_panel["has_openlocal_retail_coverage"] = validation_panel["ol_retail_unique_units"].ge(10)
validation_panel["has_openlocal_office_coverage"] = validation_panel["ol_office_unique_units"].ge(10)
validation_panel["analysis_ready_retail_validation"] = (
    validation_panel["has_gs_core_coverage"]
    & validation_panel["has_openlocal_retail_coverage"]
)
validation_panel["analysis_ready_h2_office_retail"] = (
    validation_panel["analysis_ready_retail_validation"]
    & validation_panel["has_openlocal_office_coverage"]
)

print("Merged Green Street/OpenLocal LAD-year rows:", len(validation_panel))
display(
    validation_panel.groupby("year").agg(
        lad_year_rows=("LAD24NM", "size"),
        LADs=("LAD24NM", "nunique"),
        analysis_ready_retail_validation=("analysis_ready_retail_validation", "sum"),
        analysis_ready_h2_office_retail=("analysis_ready_h2_office_retail", "sum"),
    ).reset_index()
)

display(validation_panel.head(12))

In [ ]:
def corr_summary(df, x, y, label):
    clean = df[[x, y]].dropna()
    if len(clean) < 3:
        return {
            "sample": label,
            "n": len(clean),
            "pearson": np.nan,
            "spearman": np.nan,
        }
    return {
        "sample": label,
        "n": len(clean),
        "pearson": clean.corr(method="pearson").iloc[0, 1],
        "spearman": clean.corr(method="spearman").iloc[0, 1],
    }


validation_summary = pd.DataFrame([
    corr_summary(validation_panel, "gs_weighted_vacancy", "ol_retail_vacancy_share", "all merged LAD-years"),
    corr_summary(
        validation_panel[validation_panel["analysis_ready_retail_validation"]],
        "gs_weighted_vacancy",
        "ol_retail_vacancy_share",
        "analysis-ready retail validation LAD-years",
    ),
    corr_summary(validation_panel, "gs_mean_health_index", "ol_retail_vacancy_share", "Health Index vs OpenLocal retail vacancy"),
    corr_summary(validation_panel, "gs_weighted_vacancy", "ol_retail_total_floor_area", "Green Street vacancy vs OpenLocal retail floor area"),
])

display(validation_summary)

In [ ]:
coverage_lad_summary = (
    validation_panel
    .groupby(["LAD24CD", "LAD24NM", "geocode_name"], dropna=False)
    .agg(
        years_with_both=("year", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        mean_gs_properties=("gs_properties", "mean"),
        mean_gs_units=("gs_total_units", "mean"),
        mean_openlocal_retail_units=("ol_retail_unique_units", "mean"),
        mean_openlocal_office_units=("ol_office_unique_units", "mean"),
        analysis_ready_retail_years=("analysis_ready_retail_validation", "sum"),
        analysis_ready_h2_years=("analysis_ready_h2_office_retail", "sum"),
    )
    .reset_index()
    .sort_values(["analysis_ready_retail_years", "mean_gs_properties"], ascending=[False, False])
)

display(coverage_lad_summary.head(20))
display(
    coverage_lad_summary[
        coverage_lad_summary["LAD24NM"].isin(["City of London", "Westminster"])
    ]
)

## 7. Quick Diagnostic Figures

These are source-reconciliation diagnostics, not hypothesis results. They can support the Data or Methodology chapter if needed.

In [ ]:
coverage_by_year = (
    validation_panel.groupby("year")
    .agg(
        gs_LADs=("LAD24NM", "nunique"),
        analysis_ready_retail_validation=("analysis_ready_retail_validation", "sum"),
        analysis_ready_h2_office_retail=("analysis_ready_h2_office_retail", "sum"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(coverage_by_year["year"], coverage_by_year["gs_LADs"], marker="o", label="Merged LADs")
ax.plot(coverage_by_year["year"], coverage_by_year["analysis_ready_retail_validation"], marker="o", label="Retail validation ready")
ax.plot(coverage_by_year["year"], coverage_by_year["analysis_ready_h2_office_retail"], marker="o", label="Office-retail ready")
ax.set_xlabel("Year")
ax.set_ylabel("Number of LAD-year observations")
ax.set_title("Green Street and OpenLocal Reconciliation Coverage")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "source_reconciliation_coverage_by_year.png", dpi=200)
plt.show()

plot_df = validation_panel[validation_panel["analysis_ready_retail_validation"]].dropna(
    subset=["gs_weighted_vacancy", "ol_retail_vacancy_share"]
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(plot_df["gs_weighted_vacancy"], plot_df["ol_retail_vacancy_share"], alpha=0.65)
ax.set_xlabel("Green Street weighted vacancy")
ax.set_ylabel("OpenLocal retail vacancy proxy")
ax.set_title("Source Validation: Vacancy Indicators")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "source_validation_greenstreet_vs_openlocal_vacancy.png", dpi=200)
plt.show()

## 8. Placeholder for Historical Green Street POI Churn

When the 2019-2025 historical Green Street POI flat file arrives, it should first be processed in `03_GreenStreet_POI_Preparation.ipynb`. The resulting annual opening/closure/event table can then be joined to the panels created here.

In [ ]:
prepared_dir = BASE / files["prepared_greenstreet_dir"]
possible_event_files = [
    prepared_dir / "greenstreet_historical_poi_events_2019_2025.csv",
    prepared_dir / "greenstreet_historical_poi_annual_churn_by_lad.csv",
    prepared_dir / "greenstreet_historical_poi_annual_churn_by_submarket.csv",
]

existing_event_files = [p for p in possible_event_files if p.exists()]
if existing_event_files:
    print("Historical POI event/churn outputs found:")
    for p in existing_event_files:
        print(" -", p)
else:
    print("Historical POI churn outputs are not available yet. This is expected until Green Street provides the 2019-2025 flat file.")

## 9. Export Analysis-Ready Outputs

In [ ]:
outputs = {
    "greenstreet_property_year_panel.csv": gs_property_year_panel,
    "greenstreet_lad_year_indicators.csv": gs_lad_year,
    "greenstreet_submarket_year_indicators.csv": gs_submarket_year,
    "openlocal_quarterly_snapshot_indicators.csv": openlocal_snapshot,
    "openlocal_lad_year_indicators.csv": openlocal_lad_year,
    "source_validation_lad_year_panel.csv": validation_panel,
    "source_validation_summary.csv": validation_summary,
    "source_reconciliation_coverage_by_lad.csv": coverage_lad_summary,
}

for filename, df in outputs.items():
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False)

output_manifest = pd.DataFrame([
    {
        "file": filename,
        "rows": len(df),
        "columns": len(df.columns),
        "description": {
            "greenstreet_property_year_panel.csv": "Green Street property-year vacancy, long-term vacancy and Health Index records with LAD/submarket assignment.",
            "greenstreet_lad_year_indicators.csv": "Green Street indicators aggregated to LAD-year.",
            "greenstreet_submarket_year_indicators.csv": "Green Street indicators aggregated to office-submarket-year.",
            "openlocal_quarterly_snapshot_indicators.csv": "OpenLocal quarterly retail/office indicators by LAD, used before annual averaging.",
            "openlocal_lad_year_indicators.csv": "OpenLocal annual retail/office indicators by LAD.",
            "source_validation_lad_year_panel.csv": "Merged Green Street/OpenLocal LAD-year panel for validation and acceptance checks.",
            "source_validation_summary.csv": "Correlation summaries for key validation checks.",
            "source_reconciliation_coverage_by_lad.csv": "LAD-level coverage summary across merged sources.",
        }[filename],
    }
    for filename, df in outputs.items()
])

manifest_path = OUTPUT_DIR / "manifest.csv"
output_manifest.to_csv(manifest_path, index=False)

print("Saved outputs to:", OUTPUT_DIR)
display(output_manifest)

## 10. Interpretation Notes for Dissertation Planning

- The validation panel is a data-quality and reconciliation step, not a hypothesis test.
- H1 can use Green Street vacancy/Health and OpenLocal retail adaptation indicators while the historical POI churn data are pending.
- H2 can already use OpenLocal office restructuring indicators and Green Street/OpenLocal retail outcomes.
- H3 should remain provisional until the Green Street 2019-2025 historical POI flat file is available, because annual openings/closures are needed to measure churn properly.